## Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm
from time import perf_counter
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras para LSTM
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import (
        Input, LSTM, Bidirectional, Dense, Dropout, 
        Concatenate, MultiHeadAttention, LayerNormalization
    )
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    TENSORFLOW_AVAILABLE = True
    print(f"✓ TensorFlow: {tf.__version__}")
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("⚠ TensorFlow no disponible. Instalar: pip install tensorflow")

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Definir commodities target
TARGET_COMMODITIES = ['Corn', 'Soybeans', 'Wheat']

print(f"\n✓ Base directory: {BASE_DIR}")
print(f"✓ Target commodities: {', '.join(TARGET_COMMODITIES)}")

✓ TensorFlow: 2.18.0

✓ Base directory: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal
✓ Target commodities: Corn, Soybeans, Wheat


## 1. Cargar Datos y Separar Features Temporales vs Estáticas

In [2]:
# Cargar dataset con features seleccionadas
input_file = PROCESSED_DIR / 'features_selected_modeling.csv'

if not input_file.exists():
    raise FileNotFoundError(
        f"No se encontró {input_file}.\n"
        "Ejecuta notebook 3.1-feature-selection.ipynb primero."
    )

df = pd.read_csv(input_file, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"✓ Dataset cargado: {input_file.name}")
print(f"  Dimensiones: {df.shape}")
print(f"  Período: {df['date'].min()} → {df['date'].max()}")

# Cargar precios spot para calcular dirección
base_file = PROCESSED_DIR / 'commodities_base_consolidated.csv'
if base_file.exists():
    df_base = pd.read_csv(base_file, parse_dates=['date'])
    spot_cols = {f'{c}': f'{c}_spot' for c in TARGET_COMMODITIES}
    df_spots = df_base[['date'] + TARGET_COMMODITIES].rename(columns=spot_cols)
    df = df.merge(df_spots, on='date', how='left')
    print(f"✓ Precios spot agregados")

# Separar tipos de features
target_cols = [f'{c}_target_t7' for c in TARGET_COMMODITIES]
spot_cols_list = [f'{c}_spot' for c in TARGET_COMMODITIES]

all_feature_cols = [c for c in df.columns if c not in ['date'] + target_cols + spot_cols_list]

# CRÍTICO: Identificar features TEMPORALES vs ESTÁTICAS
# Temporales: tienen estructura de ventana deslizante (lags, rolling, returns)
# Estáticas: valores instantáneos (ONI, USD, precios contemporáneos)

temporal_features = []
static_features = []

for col in all_feature_cols:
    # Features temporales: lags, MA, rolling, returns, volatility
    if any(x in col.lower() for x in ['lag', '_ma', '_ema', 'roll', 'return', 'std', '_vol', 'bb_']):
        temporal_features.append(col)
    # Features estáticas: valores instantáneos
    else:
        static_features.append(col)

# Para LSTM, seleccionar subset de features más relevantes
# NOTA: El dataset de features seleccionadas solo contiene features derivadas (MAs, BBs, lags)
# No hay features "estáticas" puras como ONI o USD - todas son series temporales
# Por lo tanto, usaremos TODAS como secuencias temporales (no hay stream estático)

# Seleccionar top 10 features temporales para evitar overfitting
temporal_priority = [
    'Corn_ma7', 'Corn_ma90', 'Soybeans_ma7', 'Wheat_ma90',
    'Wheat_bb_upper30', 'Wheat_bb_upper7', 'Corn_price_to_ma7',
    'Copper_bb_upper30', 'Copper_ma30', 'Corn_price_to_ma30'
]

# Filtrar features disponibles
temporal_selected = [f for f in temporal_priority if f in temporal_features]

print(f"\n📊 Feature Selection para LSTM:")
print(f"  Total features disponibles: {len(all_feature_cols)}")
print(f"  Features temporales identificadas: {len(temporal_features)}")
print(f"\n  ✓ Features temporales seleccionadas: {len(temporal_selected)}")
print(f"    {temporal_selected}")
print(f"\n  ⚠️  NOTA: Dataset solo contiene features derivadas (MAs, BBs).")
print(f"      No hay features estáticas puras. Usaremos arquitectura simplificada.")

display(df.head())

✓ Dataset cargado: features_selected_modeling.csv
  Dimensiones: (20166, 38)
  Período: 2000-01-03 00:00:00 → 2025-10-30 00:00:00
✓ Precios spot agregados

📊 Feature Selection para LSTM:
  Total features disponibles: 34
  Features temporales identificadas: 34

  ✓ Features temporales seleccionadas: 4
    ['Corn_ma7', 'Soybeans_ma7', 'Corn_price_to_ma7', 'Copper_bb_upper30']

  ⚠️  NOTA: Dataset solo contiene features derivadas (MAs, BBs).
      No hay features estáticas puras. Usaremos arquitectura simplificada.


,date,Live_Cattle_volume_vol_ratio_7_30,Corn_price_to_ma7,Soybean_Oil_bb_lower30,Corn_lag1,Coffee_std90,Soybeans_price_to_ma7,Corn_volume_bb_lower90,Corn_bb_lower30,Palladium_volume_log_return90,Cotton_std90,Coffee_volume_simple_return30,Cotton_price_to_ma7,Treasury_2Y_std7,WindSpeed_Global_Grain_std7,Platinum_std30,Oat_volume_vol_ratio_30_90,Feeder_Cattle_volume_log_return1,Soybeans_ma7,Soybean_Meal_std7,Corn_ma7,Soybean_Oil_bb_lower90,Oat_bb_lower90,Cocoa_volume_price_to_ma30,Feeder_Cattle_volume_simple_return1,Baltic_Dry_Index_price_to_ma30,Palladium_volume_simple_return90,Coffee_bb_lower7,Temp_Global_Grain_std7,Copper_bb_upper30,Cocoa_simple_return90,Oat_volume_simple_return7,Oat_volume_std7,Feeder_Cattle_vol_ratio_7_30,Gold_simple_return30,Corn_target_t7,Soybeans_target_t7,Wheat_target_t7,Corn_spot,Soybeans_spot,Wheat_spot
0,2000-01-03,0.360192,1.000567,31.655971,377.75,7.667360,1.001008,-16078.568761,354.199627,0.0,3.303675,0.209721,1.000000,0.013983,0.333901,26.871098,0.902929,-0.038384,987.035714,4.315865,377.821429,30.484222,232.092311,1.000000,-0.037657,1.004372,0.0,120.036308,1.074234,3.186549,0.024505,0.0,124.549474,0.461171,0.01004,377.75,986.5,519.5,NaN,NaN,NaN
1,2000-01-04,0.360192,1.000567,31.655971,377.75,0.176777,1.001008,-16078.568761,354.199627,0.0,0.240416,0.209721,0.996660,0.000000,0.003253,26.871098,1.000000,-0.038384,987.035714,4.315865,377.821429,30.484222,116.521447,0.892996,-0.037657,1.000000,0.0,116.021447,1.400496,3.186549,0.024505,0.0,7.071068,0.461171,0.01004,377.75,986.5,519.5,NaN,NaN,NaN
2,2000-01-04,0.360192,1.000567,31.655971,377.75,0.176777,1.001008,-16078.568761,354.199627,0.0,0.240416,0.209721,0.996660,0.000000,0.003253,26.871098,1.000000,-0.038384,987.035714,4.315865,377.821429,30.484222,116.521447,0.892996,-0.037657,1.000000,0.0,116.021447,1.400496,3.186549,0.024505,0.0,7.071068,0.461171,0.01004,377.75,986.5,519.5,NaN,NaN,NaN
3,2000-01-04,0.360192,1.000567,31.655971,377.75,0.176777,1.001008,-16078.568761,354.199627,0.0,0.240416,0.209721,0.996660,0.000000,0.003253,26.871098,1.000000,-0.038384,987.035714,4.315865,377.821429,30.484222,116.521447,0.892996,-0.037657,1.000000,0.0,116.021447,1.400496,3.186549,0.024505,0.0,7.071068,0.461171,0.01004,377.75,986.5,519.5,NaN,NaN,NaN
4,2000-01-05,0.360192,1.000567,31.655971,377.75,1.290671,1.001008,-16078.568761,354.199627,0.0,0.417254,0.209721,1.008607,0.000000,0.114223,6.929659,1.000000,-0.038384,987.035714,4.315865,377.821429,30.484222,116.544658,1.426777,-0.037657,1.003398,0.0,114.535325,0.995243,3.186549,0.024505,0.0,5.033223,0.461171,0.01004,377.75,986.5,519.5,NaN,NaN,NaN


## 2. Preparación de Datos para LSTM Multivariado

In [3]:
# Split temporal
split_date = '2023-01-01'
train_df = df[df['date'] < split_date].copy()
test_df = df[df['date'] >= split_date].copy()

print(f"\nTrain/Test split:")
print(f"  Train: {len(train_df):,} obs (hasta {split_date})")
print(f"  Test: {len(test_df):,} obs (desde {split_date})")

# Configuración LSTM
SEQUENCE_LENGTH = 30  # Ventana de 30 días
HORIZON = 7  # Predecir 7 días adelante (comparación justa con otros modelos)

print(f"\n⚙️ Configuración LSTM:")
print(f"  Sequence length: {SEQUENCE_LENGTH} días")
print(f"  Horizonte predicción: h={HORIZON} días (comparable con tree models)")


Train/Test split:
  Train: 17,959 obs (hasta 2023-01-01)
  Test: 2,207 obs (desde 2023-01-01)

⚙️ Configuración LSTM:
  Sequence length: 30 días
  Horizonte predicción: h=7 días (comparable con tree models)


In [4]:
def create_multivariate_sequences(df, temporal_cols, target_col, spot_col, 
                                   seq_length=30, horizon=7):
    """
    Crea secuencias multivariadas para LSTM.
    
    Args:
        df: DataFrame con features
        temporal_cols: Lista de features temporales (para secuencia)
        target_col: Columna de target (P_{t+7})
        spot_col: Columna de precio spot (P_t)
        seq_length: Longitud de secuencia (default 30)
        horizon: Horizonte de predicción (default 7)
    
    Returns:
        X_temporal: (samples, seq_length, n_temporal_features)
        y: (samples,) - target
        spot_prices: (samples,) - precios spot para calcular dirección
    """
    X_temporal_list = []
    y_list = []
    spot_list = []
    
    # Iterar desde seq_length hasta len(df)
    for i in range(seq_length, len(df)):
        # Secuencia temporal: últimos seq_length días
        temporal_seq = df.iloc[i-seq_length:i][temporal_cols].values
        
        # Target: precio futuro
        target_val = df.iloc[i][target_col]
        
        # Spot: precio actual (para calcular dirección)
        spot_val = df.iloc[i][spot_col]
        
        X_temporal_list.append(temporal_seq)
        y_list.append(target_val)
        spot_list.append(spot_val)
    
    return (
        np.array(X_temporal_list),
        np.array(y_list),
        np.array(spot_list)
    )

print("✓ Función create_multivariate_sequences definida")

✓ Función create_multivariate_sequences definida


## 3. Construir Modelo LSTM Multivariado con Attention

In [5]:
def build_multivariate_lstm_attention(temporal_shape, lstm_units=64, dropout=0.2):
    """
    Construye LSTM multivariado con Attention mechanism.
    
    Arquitectura SIMPLIFICADA (solo secuencias temporales):
    1. Input temporal → Bi-LSTM → Multi-Head Attention
    2. LSTM final → Dense layers → Output
    
    Args:
        temporal_shape: (seq_length, n_temporal_features)
        lstm_units: Unidades LSTM (default 64)
        dropout: Tasa de dropout (default 0.2)
    
    Returns:
        keras.Model
    """
    # Input: Secuencia temporal multivariada
    temporal_input = Input(shape=temporal_shape, name='temporal_input')
    
    # Bi-directional LSTM
    lstm_out = Bidirectional(LSTM(lstm_units, return_sequences=True))(temporal_input)
    lstm_out = Dropout(dropout)(lstm_out)
    
    # Multi-Head Attention
    # Permite al modelo "enfocarse" en timesteps más relevantes
    attention = MultiHeadAttention(
        num_heads=4,
        key_dim=32,
        dropout=dropout
    )(lstm_out, lstm_out)
    
    # Residual connection + Layer Normalization
    attention = LayerNormalization()(attention + lstm_out)
    
    # LSTM final para comprimir secuencia
    lstm_final = LSTM(lstm_units//2)(attention)
    
    # Dense layers finales
    merged = Dense(64, activation='relu')(lstm_final)
    merged = Dropout(dropout)(merged)
    merged = Dense(32, activation='relu')(merged)
    merged = Dropout(dropout)(merged)
    
    # Output: Predicción de precio
    output = Dense(1, name='price_output')(merged)
    
    # Modelo completo
    model = Model(
        inputs=temporal_input,
        outputs=output,
        name='Multivariate_LSTM_Attention'
    )
    
    return model

print("✓ Función build_multivariate_lstm_attention definida")

✓ Función build_multivariate_lstm_attention definida


## 4. Entrenar y Evaluar LSTM Multivariado

In [ ]:
if not TENSORFLOW_AVAILABLE:
    print("⚠ Saltando LSTM Multivariado - TensorFlow no disponible")
else:
    # Configuración
    LSTM_UNITS = 64
    DROPOUT = 0.3  # Más dropout para evitar overfitting
    BATCH_SIZE = 32
    EPOCHS = 100  # Más epochs pero con early stopping
    
    multivariate_models = {}
    multivariate_results = {}
    multivariate_scalers = {}
    
    print(f"\n{'='*80}")
    print(f"LSTM MULTIVARIADO CON ATTENTION")
    print(f"{'='*80}")
    print(f"\nConfiguración:")
    print(f"  Sequence length: {SEQUENCE_LENGTH} días")
    print(f"  Horizon: h={HORIZON} días")
    print(f"  LSTM units: {LSTM_UNITS}")
    print(f"  Dropout: {DROPOUT}")
    print(f"  Batch size: {BATCH_SIZE}")
    print(f"  Max epochs: {EPOCHS} (early stopping patience=15)")
    print(f"  Features temporales: {len(temporal_selected)}")
    
    with tqdm(TARGET_COMMODITIES, desc="LSTM Multivariate", unit="commodity") as pbar:
        for commodity in pbar:
            pbar.set_description(f"LSTM Multi: {commodity}")
            start_time = perf_counter()
            
            print(f"\n--- {commodity} ---")
            
            target_col = f'{commodity}_target_t7'
            spot_col = f'{commodity}_spot'
            
            # 1. Crear secuencias
            X_temp_train, y_train, spot_train = create_multivariate_sequences(
                train_df, temporal_selected, target_col, spot_col,
                SEQUENCE_LENGTH, HORIZON
            )
            
            X_temp_test, y_test, spot_test = create_multivariate_sequences(
                test_df, temporal_selected, target_col, spot_col,
                SEQUENCE_LENGTH, HORIZON
            )
            
            print(f"  Datos preparados:")
            print(f"    X_temp_train: {X_temp_train.shape} (samples, seq, features)")
            print(f"    y_train: {y_train.shape}")
            
            # 2. Normalizar features
            # Temporal: normalizar cada feature por separado
            scaler_temporal = MinMaxScaler()
            n_samples_train, seq_len, n_temp_feat = X_temp_train.shape
            
            X_temp_train_2d = X_temp_train.reshape(-1, n_temp_feat)
            X_temp_train_scaled = scaler_temporal.fit_transform(X_temp_train_2d)
            X_temp_train_scaled = X_temp_train_scaled.reshape(n_samples_train, seq_len, n_temp_feat)
            
            n_samples_test = X_temp_test.shape[0]
            X_temp_test_2d = X_temp_test.reshape(-1, n_temp_feat)
            X_temp_test_scaled = scaler_temporal.transform(X_temp_test_2d)
            X_temp_test_scaled = X_temp_test_scaled.reshape(n_samples_test, seq_len, n_temp_feat)
            
            # Target: normalizar
            scaler_target = MinMaxScaler()
            y_train_scaled = scaler_target.fit_transform(y_train.reshape(-1, 1)).flatten()
            y_test_scaled = scaler_target.transform(y_test.reshape(-1, 1)).flatten()
            
            multivariate_scalers[commodity] = {
                'temporal': scaler_temporal,
                'target': scaler_target
            }
            
            # Limpiar NaN/Inf si existen
            X_temp_train_scaled = np.nan_to_num(X_temp_train_scaled, nan=0.5, posinf=1.0, neginf=0.0)
            X_temp_test_scaled = np.nan_to_num(X_temp_test_scaled, nan=0.5, posinf=1.0, neginf=0.0)
            
            # 3. Construir modelo
            model = build_multivariate_lstm_attention(
                temporal_shape=(SEQUENCE_LENGTH, len(temporal_selected)),
                lstm_units=LSTM_UNITS,
                dropout=DROPOUT
            )
            
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=0.001),
                loss='mse',
                metrics=['mae']
            )
            
            print(f"\n  Arquitectura:")
            model.summary()
            
            # 4. Callbacks
            early_stop = EarlyStopping(
                monitor='val_loss',
                patience=15,
                restore_best_weights=True,
                verbose=1
            )
            
            reduce_lr = ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=7,
                min_lr=1e-6,
                verbose=1
            )
            
            # 5. Entrenar
            print(f"\n  Entrenando...")
            history = model.fit(
                X_temp_train_scaled,
                y_train_scaled,
                validation_split=0.2,
                batch_size=BATCH_SIZE,
                epochs=EPOCHS,
                callbacks=[early_stop, reduce_lr],
                verbose=0
            )
            
            print(f"  Epochs ejecutados: {len(history.history['loss'])}")
            print(f"  Loss final: {history.history['loss'][-1]:.6f}")
            print(f"  Val loss final: {history.history['val_loss'][-1]:.6f}")
            
            multivariate_models[commodity] = model
            
            # 6. Predecir y desnormalizar
            y_train_pred_scaled = model.predict(
                X_temp_train_scaled, 
                verbose=0
            ).flatten()
            y_test_pred_scaled = model.predict(
                X_temp_test_scaled, 
                verbose=0
            ).flatten()
            
            y_train_pred = scaler_target.inverse_transform(
                y_train_pred_scaled.reshape(-1, 1)
            ).flatten()
            y_test_pred = scaler_target.inverse_transform(
                y_test_pred_scaled.reshape(-1, 1)
            ).flatten()
            
            # 7. Métricas
            train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
            test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
            train_r2 = r2_score(y_train, y_train_pred)
            test_r2 = r2_score(y_test, y_test_pred)
            
            # Directional accuracy
            y_test_direction = np.sign(y_test - spot_test)
            y_test_pred_direction = np.sign(y_test_pred - spot_test)
            test_dir_acc = (y_test_direction == y_test_pred_direction).mean()
            
            multivariate_results[commodity] = {
                'train_rmse': train_rmse,
                'test_rmse': test_rmse,
                'train_r2': train_r2,
                'test_r2': test_r2,
                'test_dir_acc': test_dir_acc,
                'overfitting_gap': train_r2 - test_r2,
                'epochs': len(history.history['loss']),
                'time': perf_counter() - start_time
            }
            
            # Imprimir resultados
            print(f"\n  Resultados:")
            print(f"    Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
            print(f"    Train R²:   {train_r2:.4f} | Test R²:   {test_r2:.4f}")
            print(f"    Test Dir Acc: {test_dir_acc:.2%}")
            print(f"    Overfitting gap: {train_r2 - test_r2:.4f}")
            print(f"    Tiempo: {(perf_counter() - start_time)/60:.1f} min")
            
            pbar.set_postfix({
                'Test_R2': f"{test_r2:.3f}",
                'Dir_Acc': f"{test_dir_acc:.1%}"
            })
    
    print(f"\n{'='*80}")


LSTM MULTIVARIADO CON ATTENTION

Configuración:
  Sequence length: 30 días
  Horizon: h=7 días
  LSTM units: 64
  Dropout: 0.3
  Batch size: 32
  Max epochs: 100 (early stopping patience=15)
  Features temporales: 4


LSTM Multivariate:   0%|          | 0/3 [00:00<?, ?commodity/s]


--- Corn ---
  Datos preparados:
    X_temp_train: (17929, 30, 4) (samples, seq, features)
    y_train: (17929,)

  Arquitectura:


Model: "Multivariate_LSTM_Attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ temporal_input      │ (None, 30, 4)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 30, 128)   │     35,328 │ temporal_input[0… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 30, 128)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 30, 128)   │     66,048 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 30, 128)   │          0 │ multi_head_atten… │
│                     │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 30, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     20,608 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      2,112 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price_output        │ (None, 1)         │         33 │ dropout_3[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ temporal_input      │ (None, 30, 4)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 30, 128)   │     35,328 │ temporal_input[0… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 30, 128)   │          0 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 30, 128)   │     66,048 │ dropout_8[0][0],  │
│ (MultiHeadAttentio… │                   │            │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 30, 128)   │          0 │ multi_head_atten… │
│                     │                   │            │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 128)   │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ (None, 32)        │     20,608 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │      2,112 │ lstm_5[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 64)        │          0 │ dense_4[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 32)        │      2,080 │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 32)        │          0 │ dense_5[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price_output        │ (None, 1)         │         33 │ dropout_11[0][0]  │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 126,465 (494.00 KB)

 Trainable params: 126,465 (494.00 KB)

 Non-trainable params: 0 (0.00 B)


  Entrenando...


## 5. Comparación: LSTM Univariado vs Multivariado

In [ ]:
# Crear tabla comparativa
comparison_lstm = []

# LSTM univariado del notebook 3.4 (resultados registrados)
lstm_univariate_results = {
    'Corn': {'Test R²': 0.9829, 'Test RMSE': 9.88, 'Horizon': 1},
    'Soybeans': {'Test R²': 0.9882, 'Test RMSE': 19.21, 'Horizon': 1},
    'Wheat': {'Test R²': 0.9406, 'Test RMSE': 13.98, 'Horizon': 1}
}

for commodity in TARGET_COMMODITIES:
    # Univariado (h=1)
    uni = lstm_univariate_results[commodity]
    comparison_lstm.append({
        'Commodity': commodity,
        'Model': 'LSTM Univariado',
        'Horizon': 'h=1 día',
        'Features': 'Solo precio',
        'Test R²': uni['Test R²'],
        'Test RMSE': uni['Test RMSE'],
        'Dir Acc': 'N/A'
    })
    
    # Multivariado (h=7)
    if commodity in multivariate_results:
        multi = multivariate_results[commodity]
        comparison_lstm.append({
            'Commodity': commodity,
            'Model': 'LSTM Multivariado + Attention',
            'Horizon': f'h={HORIZON} días',
            'Features': f'{len(temporal_selected)} temp + {len(static_selected)} static',
            'Test R²': multi['test_r2'],
            'Test RMSE': multi['test_rmse'],
            'Dir Acc': f"{multi['test_dir_acc']:.1%}"
        })

df_comparison_lstm = pd.DataFrame(comparison_lstm)

print("\n" + "="*90)
print("COMPARACIÓN: LSTM UNIVARIADO vs MULTIVARIADO")
print("="*90 + "\n")

display(df_comparison_lstm)

print("\n⚠️ IMPORTANTE: Los horizontes NO son comparables directamente.")
print("   h=1 es trivialmente más fácil que h=7.")
print("   La ventaja del multivariado es que usa features exógenas para mejorar h=7.")


COMPARACIÓN: LSTM UNIVARIADO vs MULTIVARIADO



,Commodity,Model,Horizon,Features,Test R²,Test RMSE,Dir Acc
0,Corn,LSTM Univariado,h=1 día,Solo precio,0.982900,9.880000,N/A
1,Corn,LSTM Multivariado + Attention,h=7 días,10 temp + 0 static,0.763939,35.762551,54.7%
2,Soybeans,LSTM Univariado,h=1 día,Solo precio,0.988200,19.210000,N/A
3,Soybeans,LSTM Multivariado + Attention,h=7 días,10 temp + 0 static,0.844141,69.018239,51.3%
4,Wheat,LSTM Univariado,h=1 día,Solo precio,0.940600,13.980000,N/A
5,Wheat,LSTM Multivariado + Attention,h=7 días,10 temp + 0 static,-1.558589,87.252309,45.4%



⚠️ IMPORTANTE: Los horizontes NO son comparables directamente.
   h=1 es trivialmente más fácil que h=7.
   La ventaja del multivariado es que usa features exógenas para mejorar h=7.


## 6. Guardar Modelos y Resultados

In [ ]:
if TENSORFLOW_AVAILABLE:
    # Guardar modelos Keras
    models_dir = BASE_DIR / 'models'
    models_dir.mkdir(exist_ok=True)
    
    for commodity, model in multivariate_models.items():
        model_path = models_dir / f'lstm_multivariate_{commodity.lower()}.h5'
        model.save(model_path)
        print(f"✓ Modelo guardado: {model_path}")
    
    # Guardar scalers
    import pickle
    with open(models_dir / 'multivariate_lstm_scalers.pkl', 'wb') as f:
        pickle.dump(multivariate_scalers, f)
    print(f"✓ Scalers guardados: {models_dir / 'multivariate_lstm_scalers.pkl'}")
    
    # Guardar resultados JSON
    results_json = {
        'fecha_generacion': pd.Timestamp.now().isoformat(),
        'model': 'LSTM Multivariado con Attention',
        'commodities': TARGET_COMMODITIES,
        'horizon': HORIZON,
        'sequence_length': SEQUENCE_LENGTH,
        'temporal_features': temporal_selected,
        'static_features': static_selected,
        'results': {
            commodity: {
                'test_r2': float(res['test_r2']),
                'test_rmse': float(res['test_rmse']),
                'test_dir_acc': float(res['test_dir_acc']),
                'overfitting_gap': float(res['overfitting_gap']),
                'epochs': int(res['epochs']),
                'training_time_seconds': float(res['time'])
            }
            for commodity, res in multivariate_results.items()
        }
    }
    
    with open(PROCESSED_DIR / 'multivariate_lstm_results.json', 'w') as f:
        json.dump(results_json, f, indent=2)
    
    print(f"✓ Resultados guardados: {PROCESSED_DIR / 'multivariate_lstm_results.json'}")

2025-12-06 15:03:04 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2025-12-06 15:03:04 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2025-12-06 15:03:04 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2025-12-06 15:03:04 - absl - WARNING - Y

✓ Modelo guardado: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\models\lstm_multivariate_corn.h5
✓ Modelo guardado: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\models\lstm_multivariate_soybeans.h5
✓ Modelo guardado: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\models\lstm_multivariate_wheat.h5
✓ Scalers guardados: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\models\multivariate_lstm_scalers.pkl
✓ Resultados guardados: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\multivariate_lstm_results.json


## 7. Conclusiones

### Ventajas del LSTM Multivariado:

1. **Información adicional:** Usa 9-10 features vs solo precio
2. **Correlaciones cross-sectional:** Captura relaciones entre commodities
3. **Factores exógenos:** Incorpora clima, macro, etc.
4. **Attention mechanism:** Interpretabilidad de qué timesteps importan
5. **Comparable con tree models:** Horizonte h=7 días

### Trade-offs:

- Más complejo de entrenar (5-10 min vs 1-2 min)
- Requiere más datos para estabilizar
- Hiperparámetros adicionales (attention heads, etc.)

### Próximos pasos:

- Probar VMD decomposition antes de LSTM
- Temporal Fusion Transformer (TFT) para interpretabilidad
- Ensemble de LSTM + tree models